In [ ]:
#LLM 종류별 성능평가.

1. 환경설정
- 데이터셋 불러오기 : tool_execution_test_Dataset/test_dataset_cleaned.csv 사용
- 프롬프트 준비 : tool_node에서 사용할 SYSTEM_PROMPT 준비.
- LLM API KEY 준비 : .env에 작성되어있는 Gemini. Claude, OpenAI 키 준비.
- LangGraph 준비 : `1. tool_execution_test_Dataset_Generation.ipynb`의 `2.ToolNode` 부분에 나와있는 Graph 준비. 실제 Tool Call은 없이 llm_with_tools를 사용하는 Tool Node만 사용.


2. 평가 실행
데이터셋 `content`칼럼을 1행씩 graph에 넣고 실행. 최종 결과는 model_evaluation_result.csv에 `모델명_tool_called`, `모델명_tool_name`, `모델명_query`로 저장. 10행 실행시마다 처리 내용을 저장하는 백업 기능 구현.
위 평가를 지정한 전체 LLM 모델 별로 진행.
모델 : gpt-4o-mini, gpt-4o, gpt-5, gemini-3.1-pro-preview, gemini-3.5-flash, gemini-3-flash-preview, claude-opus-4-7, claude-sonnet-4-6, claude-haiku-4-5-20251001
temperature = 0

3. 시각화
데이터셋과 평가 결과 대조하여 결과 시각화.
LLM별 tool_called, tool_name 성능지표 : accuracy, recall, f1-score
LLM별 query 품질볖가 : LLM-as-a-Judge (gpt-4o). 테스트 데이터셋 context와 정답 query, 모델별 query를 대조해 1~5점으로 점수 작성. 점수별 기준 구체적으로 작성.


In [ ]:
  ┌──────────────────┬──────────────────────────────────────────┐
  │        셀        │                   내용                   │
  ├──────────────────┼──────────────────────────────────────────┤
  │ 기존 1           │ 스펙 메모
  ├──────────────────┼──────────────────────────────────────────┤
  │ 기존 2           │ SYSTEM_PROMPT                            │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + pip install    │ 필요 패키지 설치                         │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + imports & keys │ import, .env 로드, 데이터셋 로드         │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + tools & graph  │ 도구 정의, ToolState, build_eval_graph() │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + MODELS         │ 9개 모델 딕셔너리                        │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + 평가 실행      │ 전체 루프 + 10행마다 백업 + CSV 저장     │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + 시각화 헤더    │ Markdown                                 │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + 성능 지표      │ accuracy/recall/F1 계산 + 2×3 bar chart  │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + Judge 헤더     │ Markdown                                 │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + Judge 설정     │ JUDGE_SYSTEM_PROMPT, judge_query()       │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + Judge 실행     │ 루프 + query_score/reason 컬럼 추가      │
  ├──────────────────┼──────────────────────────────────────────┤
  │ + Judge 시각화   │ 평균 점수 bar + 점수 분포 stacked bar    │
  └──────────────────┴──────────────────────────────────────────┘

In [6]:
SYSTEM_PROMPT = """
    #역할
    당신은 검색 전문가입니다. 입력은 현재 PPT 슬라이드의 전체 내용을 빠짐없이 적은 것입니다.
    이를 읽고 도구 선택 기준에 맞으면 도구를 사용해 검색해주세요.
    사용할 수 있는 도구는 다음과 같습니다. : tavily_search, arxiv_search
    구체적인 내용 없이 전체 내용이 짧은 경우 표지, 목차, 섹션 구분 등에 해당합니다.

    # 도구 선택 기준
    1. tavily_search : 특정 서비스와 제품에 대한 정보가 필요할 때
    특정 제품/서비스가 명시되어 있고 그에 대한 구체적인 정보(기능, 성능, 비교 등) 내용을 담은 슬라이드일 때 해당 제품에 대한 최신 정보와 동향을 검색.

    - tavily 쿼리 지침
    쿼리는 핵심 키워드만 사용해 간결하게 작성하세요.
    슬라이드에 나와 있는 핵심 제품 한 가지에 대해서만 검색하세요.
    검색어는 3~5단어면 충분합니다.

    검색 필요 예시)
    MAAL (Multilingual Adaptive Augmentation Language-model) 한국어에 강한 언어 생성 모델 MAAL을 기반으로 On-premise LLM 솔루션을 제공합니다. 다양한 파라미터 수의 모델(8B-70B)을 기반으로 고객의 니즈에 맞는 모델을 지원합니다... On-premise용으로는 성능이 뛰어나며 범용적으로 사용하기 좋은 MAAL-albatross(70B)를 권장하고 있습니다. -> MAAL 관련 검색
    Agados UI, Flow Design & Visibility Technologies Structure of this presentation Application을 위한 Architecture - SW Package를 위한 Smart Architecture - Hybrid Architecture Overview - 타 시스템과의 Interface... -> Agados 관련 검색

    2. arxiv_search : 연구에 대한 구체적인 성능 내용이 필요할 때
    연구, 성능 그래프, 성능 표, 논문 인용 표기 등이 명시되어 있을 때 그와 관련한 논문 검색.

    - arxiv 쿼리 지침
    슬라이드에 나와 있는 핵심적인 것 단 한가지에 대해서만 검색하세요.

    검색 필요 예시)
    - "REPLUG: Retrieval-Augmented Black-Box Language Models, NAACL24'", "Dense Passage Retrieval for Open-Domain Question Answering, EMNLP20'" -> 논문 검색
    - 마크다운 형식으로 전환된 그래프의 성능, 메트릭 표 -> 논문 검색
    - 특정 모델의 벤치마크 점수가 수치로 제시된 경우 예) "MAAL 70B: 9.06 / GPT-4o: 9.59" 처럼 모델별 점수 비교표가 포함된 슬라이드 -> 논문 검색
    - LogicKor, KoBEST, MMLU 등 평가 지표명이 명시된 경우

    3. 검색이 필요 없는 경우
    그 외 아래의 경우에 해당할 경우 '검색 필요 없음'과 그 이유를 출력하세요
    - 형식적 내용 : 표지, 개요, 목차, 섹션 구분, 마지막 페이지(Q&A, 감사합니다) 등의 슬라이드로 판단되는 경우
    - 소개 : 학습 목표, 강사 소개, 참고문헌 목록, 레퍼런스 목록 등을 소개하는 내용인 경우.
    - 그 외 어떠한 도구 선택 기준의 경우에도 해당하지 않는 경우.

    검색 불필요 예시)
    - 제목: 제목 없음\n- Chapter 2. 오픈소스 컨설팅의 On-premise LLM 솔루션\n- MAAL (Multilingual Adaptive Augmentation Language-model) MAAL 기반 On-premise 패키지\n  - 챗봇\n  - Chatplay\n  - LLM Task UI\n- 표: 없음 -> 목차 슬라이드
    Biz. Application을 위한 디자이너/재조정기\n‘아가도스’는 귀사의 SW Application내에서 Configure Tool의 역할 수행\n19\nⒸ 2014 agados All rights reserved. -> 섹션 구분
    제목: Jamcracker 소개 시작\n\nCloud Management Platform & Cloud Service Brokerage\n\n- CLOUD SERVICES BROKERAGE\n- CLOUD GOVERNANCE\n- MICROSOFT CSP ENABLEMENT\n- HYBRID CLOUD MANAGEMENT\n- Microsoft Cloud Solution Provider\n- OSC ASIA GROUP LIMITED\n- @ OSC Korea & OSC Asia Group jerry@osckorea.com jerry@oscasia.net +82 10 9196 1416 -> 목차 슬라이드

    # 규칙
    - 검색 시 하나의 도구만 사용하세요.
    - 대화 로그를 보았을 때 이미 검색을 진행했다면 추가 검색을 하지 말고 검색 결과를 전체 정리해주세요.
    - 검색 결과 정리시 핵심 내용을 정리해 작성해주세요.
    - 검색 결과 요약 시 검색 결과를 제외한 다른 어떠한 출력도 하지 마세요.
"""

In [7]:
!pip install langchain-openai langchain-anthropic langchain-google-genai langchain-community langchain-tavily tavily-python arxiv python-dotenv scikit-learn matplotlib seaborn pandas langgraph -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os, json, time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Annotated, TypedDict, List

from dotenv import dotenv_values
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_tavily import TavilySearch
from langchain.tools import tool
import arxiv

from sklearn.metrics import accuracy_score, recall_score, f1_score
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# ── API 키 로드 ────────────────────────────────────────────────────────────────
config = dotenv_values('./.env')
os.environ['OPENAI_API_KEY']    = config.get('OPENAI_API_KEY',    '').strip()
os.environ['TAVILY_API_KEY']    = config.get('TAVILY_API_KEY',    '').strip()
os.environ['ANTHROPIC_API_KEY'] = config.get('ANTHROPIC_API_KEY', '').strip()
os.environ['GOOGLE_API_KEY']    = config.get('GOOGLE_API_KEY',    '').strip()

for name, key in [('OpenAI',    os.environ['OPENAI_API_KEY']),
                  ('Tavily',    os.environ['TAVILY_API_KEY']),
                  ('Anthropic', os.environ['ANTHROPIC_API_KEY']),
                  ('Google',    os.environ['GOOGLE_API_KEY'])]:
    status = f"{key[:15]}...{key[-4:]}" if len(key) > 19 else ("로드됨" if key else "❌ 없음")
    print(f"{name:10s}: {status}")

# ── 데이터셋 로드 ──────────────────────────────────────────────────────────────
df = pd.read_csv('./tool_execution_test_Dataset/test_dataset_cleaned.csv')
print(f"\n데이터셋 로드: {len(df)}행  |  컬럼: {list(df.columns)}")
df.head(2)

OpenAI    : sk-proj-fk0_o2Z...MUsA
Tavily    : tvly-dev-3LfrYm...qjfK
Anthropic : sk-ant-api03-WX...yAAA
Google    : AIzaSyCJS9NBMjh...5Lk0

데이터셋 로드: 128행  |  컬럼: ['slide_index', 'title', 'content', 'tool_called', 'tool_name', 'query', 'raw_tool_result', 'tool_node_result', 'tool_called_correct', 'tool_name_correct', 'query_correct', 'judge_reason']


,slide_index,title,content,tool_called,tool_name,query,raw_tool_result,tool_node_result,tool_called_correct,tool_name_correct,query_correct,judge_reason
0,1,제목 없음,"- 제목: 제목 없음\n- 본문 텍스트: """", """", """", """", """"\n- 표...",False,none,NaN,NaN,"검색 필요 없음: 표지 슬라이드로 판단됩니다. 구체적인 제품/서비스 기능, 성능, ...",True,True,NaN,슬라이드가 목차나 개요에 해당하여 검색이 필요하지 않음
1,2,제목 없음,- 제목: 제목 없음\n\n- 텍스트(추출 순서 그대로)\n 1) [공란]\n ...,False,none,NaN,NaN,"검색 필요 없음: 제품 차별성, 사내 On-premise LLM 필요성, On-pr...",True,True,NaN,슬라이드는 목차에 해당하며 검색이 필요하지 않습니다.


In [9]:
# ── 도구 정의 (Dataset_Generation 노트북과 동일) ───────────────────────────────
tavily_tool = TavilySearch(max_results=3)

@tool
def arxiv_tool(query: str) -> str:
    """학술 논문, 연구 결과, 알고리즘의 이론적 근거를 찾을 때 사용"""
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=1)
    results = []
    for paper in client.results(search):
        results.append(
            f"제목: {paper.title}\n"
            f"연도: {paper.published.year}\n"
            f"요약: {paper.summary}\n"
            f"URL: {paper.entry_id}"
        )
    return "\n\n".join(results) if results else "결과 없음"

tool_list = [tavily_tool, arxiv_tool]

# ── State 정의 ─────────────────────────────────────────────────────────────────
class ToolState(TypedDict, total=False):
    cur_page_content: str
    cur_search_context: str
    messages: Annotated[list, add_messages]

# ── 평가용 그래프 빌더 (tool_node 없이 LLM 판단 단계까지만) ───────────────────
def build_eval_graph(llm):
    """
    tool_search 노드만 실행하고 END로 종료.
    실제 Tavily/arxiv API 호출 없이 LLM의 tool_calls 결과만 수집.
    """
    llm_with_tools = llm.bind_tools(tool_list)

    def tool_search(state: ToolState) -> dict:
        prior_log = "\n\n".join([m.content or "" for m in state.get("messages", [])])
        human_msg = f"[슬라이드 내용]\n{state.get('cur_page_content', '')}\n[대화 로그]\n{prior_log}"
        result = llm_with_tools.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=human_msg)
        ])
        return {"cur_search_context": result.content, "messages": [result]}

    builder = StateGraph(ToolState)
    builder.add_node("tool_search", tool_search)
    builder.add_edge(START, "tool_search")
    builder.add_edge("tool_search", END)   # tool_node 없이 바로 종료

    return builder.compile()

print("그래프 빌더 준비 완료")

그래프 빌더 준비 완료


In [10]:
MODELS = {
    "gpt-4o-mini":             ChatOpenAI(model="gpt-4o-mini",             temperature=0),
    "gpt-4o":                  ChatOpenAI(model="gpt-4o",                  temperature=0),
    "gpt-5":                   ChatOpenAI(model="gpt-5",                   temperature=0),
    "gemini-3.1-pro-preview":  ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview",  temperature=0),
    "gemini-3.5-flash":        ChatGoogleGenerativeAI(model="gemini-3.5-flash",        temperature=0),
    "gemini-3-flash-preview":  ChatGoogleGenerativeAI(model="gemini-3-flash-preview",  temperature=0),
    "claude-opus-4-7":         ChatAnthropic(model="claude-opus-4-7",         temperature=0),
    "claude-sonnet-4-6":       ChatAnthropic(model="claude-sonnet-4-6",       temperature=0),
    "claude-haiku-4-5":        ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0),
}

MODEL_NAMES = list(MODELS.keys())
print(f"평가 모델 {len(MODELS)}개:", MODEL_NAMES)

평가 모델 9개: ['gpt-4o-mini', 'gpt-4o', 'gpt-5', 'gemini-3.1-pro-preview', 'gemini-3.5-flash', 'gemini-3-flash-preview', 'claude-opus-4-7', 'claude-sonnet-4-6', 'claude-haiku-4-5']


In [15]:
RESULT_CSV = './tool_execution_test_Dataset/model_evaluation_result.csv'

# ── 기존 결과 로드 (있으면) ────────────────────────────────────────────────────
if Path(RESULT_CSV).exists():
    df_result = pd.read_csv(RESULT_CSV)
    print(f"기존 결과 로드: {RESULT_CSV}  ({len(df_result)}행 × {len(df_result.columns)}열)")
else:
    df_result = df[['slide_index', 'title', 'content', 'tool_called', 'tool_name', 'query']].copy()
    print("새로 시작")

# ── 모델별 평가 루프 ───────────────────────────────────────────────────────────
for model_name, model in MODELS.items():
    col_tc = f"{model_name}_tool_called"
    col_tn = f"{model_name}_tool_name"
    col_q  = f"{model_name}_query"

    # 완료 여부 및 재개 위치 확인
    if col_tc in df_result.columns and not df_result[col_tc].isna().all():
        nan_mask = df_result[col_tc].isna()
        if not nan_mask.any():
            print(f"⏭  {model_name}: 완료 — 건너뜀")
            continue
        start_i = int(nan_mask.idxmax())   # 첫 번째 NaN 위치 = 재개 지점
        tc_list = df_result[col_tc].iloc[:start_i].tolist()
        tn_list = df_result[col_tn].iloc[:start_i].tolist()
        q_list  = df_result[col_q].iloc[:start_i].tolist()
        print(f"\n{'='*60}")
        print(f"🔄 {model_name}: {start_i}행부터 재개")
        print(f"{'='*60}")
    else:
        start_i = 0
        tc_list, tn_list, q_list = [], [], []
        print(f"\n{'='*60}")
        print(f"▶  {model_name}: 처음부터 시작")
        print(f"{'='*60}")

    # 그래프 빌드
    try:
        graph = build_eval_graph(model)
    except Exception as e:
        print(f"  [ERROR] 그래프 빌드 실패: {e}")
        continue

    # start_i 행부터 실행
    for idx, row in df.iloc[start_i:].iterrows():
        try:
            result      = graph.invoke({'cur_page_content': row['content'], 'messages': []})
            last_msg    = result['messages'][-1]
            tool_called = bool(last_msg.tool_calls)
            tool_name   = last_msg.tool_calls[0]['name'] if tool_called else 'none'
            query       = last_msg.tool_calls[0]['args'].get('query', '') if tool_called else None
        except Exception as e:
            print(f"  [ERROR] row={idx}: {e}")
            tool_called, tool_name, query = None, None, None

        tc_list.append(tool_called)
        tn_list.append(tool_name)
        q_list.append(query)

        done = len(tc_list)
        q_preview = str(query)[:40] if query else '-'
        print(f"  [{done:3d}/{len(df)}] row={idx:3d} | tool_called={tool_called} | tool_name={tool_name} | query={q_preview}")

        # 10행마다 현재까지 결과를 CSV에 저장 (재개 포인트)
        if done % 10 == 0:
            pad = [None] * (len(df) - done)
            df_result[col_tc] = tc_list + pad
            df_result[col_tn] = tn_list + pad
            df_result[col_q]  = q_list  + pad
            df_result.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')
            print(f"  💾 {done}행 저장 → {RESULT_CSV}")

        time.sleep(3)   # 행 간 처리 간격

    # 모델 완료 후 전체 저장
    df_result[col_tc] = tc_list
    df_result[col_tn] = tn_list
    df_result[col_q]  = q_list
    df_result.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')
    print(f"✅ {model_name} 완료")

print(f"\n전체 완료 → {RESULT_CSV}  ({len(df_result)}행 × {len(df_result.columns)}열)")

기존 결과 로드: ./tool_execution_test_Dataset/model_evaluation_result.csv  (128행 × 12열)
⏭  gpt-4o-mini: 완료 — 건너뜀

🔄 gpt-4o: 27행부터 재개
  [ 28/128] row= 27 | tool_called=False | tool_name=none | query=-
  [ 29/128] row= 28 | tool_called=False | tool_name=none | query=-
  [ 30/128] row= 29 | tool_called=False | tool_name=none | query=-
  💾 30행 저장 → ./tool_execution_test_Dataset/model_evaluation_result.csv
  [ 31/128] row= 30 | tool_called=True | tool_name=tavily_search | query=MAAL Multilingual Adaptive Augmentation 
  [ 32/128] row= 31 | tool_called=True | tool_name=tavily_search | query=MAAL On-premise LLM 솔루션
  [ 33/128] row= 32 | tool_called=True | tool_name=tavily_search | query=MAAL On-premise LLM 솔루션
  [ 34/128] row= 33 | tool_called=False | tool_name=none | query=-
  [ 35/128] row= 34 | tool_called=True | tool_name=tavily_search | query=MAAL On-premise LLM 솔루션
  [ 36/128] row= 35 | tool_called=True | tool_name=tavily_search | query=MAAL On-premise LLM 솔루션
  [ 37/128] row= 36 | tool_calle

KeyboardInterrupt: 

# 시각화 — tool_called / tool_name 성능 지표

In [ ]:
df_result = pd.read_csv('./tool_execution_test_Dataset/model_evaluation_result.csv')

y_true_tc = df_result['tool_called'].tolist()
y_true_tn = df_result['tool_name'].tolist()

def calc_metrics(y_true, y_pred, task='binary'):
    pairs = [(t, p) for t, p in zip(y_true, y_pred)
             if p is not None and str(p) != 'nan']
    if not pairs:
        return {'accuracy': None, 'recall': None, 'f1': None}
    yt, yp = zip(*pairs)
    if task == 'binary':
        return {
            'accuracy': accuracy_score(yt, yp),
            'recall':   recall_score(yt, yp, zero_division=0),
            'f1':       f1_score(yt, yp, zero_division=0),
        }
    else:
        return {
            'accuracy': accuracy_score(yt, yp),
            'recall':   recall_score(yt, yp, average='macro', zero_division=0),
            'f1':       f1_score(yt, yp, average='macro', zero_division=0),
        }

metrics_tc, metrics_tn = {}, {}
for m in MODEL_NAMES:
    metrics_tc[m] = calc_metrics(y_true_tc, df_result[f'{m}_tool_called'].tolist(), 'binary')
    metrics_tn[m] = calc_metrics(y_true_tn, df_result[f'{m}_tool_name'].tolist(),   'multiclass')

df_metrics_tc = pd.DataFrame(metrics_tc).T.round(4)
df_metrics_tn = pd.DataFrame(metrics_tn).T.round(4)

print("=== tool_called (binary) ===")
print(df_metrics_tc.to_string())
print("\n=== tool_name (macro) ===")
print(df_metrics_tn.to_string())

# ── 시각화 ─────────────────────────────────────────────────────────────────────
model_labels = [m.replace('gemini-', 'gem-').replace('claude-', 'cl-') for m in MODEL_NAMES]
metric_cols  = ['accuracy', 'recall', 'f1']
metric_labels = ['Accuracy', 'Recall', 'F1-Score']
colors = {'tool_called': 'steelblue', 'tool_name': 'coral'}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('LLM별 Tool 선택 성능 비교', fontsize=15, fontweight='bold', y=1.01)

for row_i, (metrics_dict, task_label, color) in enumerate([
    (metrics_tc, 'tool_called (binary)',  'steelblue'),
    (metrics_tn, 'tool_name (macro)',     'coral'),
]):
    for col_j, (metric, mlabel) in enumerate(zip(metric_cols, metric_labels)):
        ax = axes[row_i][col_j]
        vals = [metrics_dict[m][metric] or 0 for m in MODEL_NAMES]
        bars = ax.bar(model_labels, vals, color=color, edgecolor='white', linewidth=0.5)
        ax.set_title(f'{task_label}\n{mlabel}', fontsize=11)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel('Score')
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('./tool_execution_test_Dataset/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# LLM-as-a-Judge — Query 품질 평가

In [ ]:
JUDGE_SYSTEM_PROMPT = """당신은 검색 쿼리 품질 평가 전문가입니다.
아래는 PPT 슬라이드 내용과 해당 슬라이드에 대한 정답 검색 쿼리, 그리고 평가 대상 쿼리입니다.
정답 쿼리를 참고하여 평가 대상 쿼리의 품질을 1~5점으로 평가하세요.

# 채점 기준
1점: 슬라이드 내용과 무관하거나 완전히 잘못된 키워드를 사용한 쿼리. 검색 시 관련 결과가 나올 가능성이 거의 없음.
2점: 관련 키워드를 일부 포함하나 핵심 주제를 벗어났거나, 불필요한 단어가 많아 검색 노이즈가 발생할 수 있음.
3점: 핵심 키워드를 포함하지만 범위가 너무 넓거나 좁음. 또는 검색어가 지나치게 길어 간결성이 부족함.
4점: 핵심 키워드를 잘 포함하고 간결하며 검색에 효과적. 정답 쿼리와 의미적으로 유사한 수준.
5점: 정답 쿼리와 동일하거나 그보다 더 정확하고 간결한 최적의 쿼리. 검색 결과의 품질이 충분히 보장됨.

# 출력 형식
반드시 아래 JSON만 출력하세요. 다른 텍스트는 절대 출력하지 마세요.
{"score": 1~5 정수, "reason": "한 문장 평가 이유"}"""

judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)

def judge_query(content: str, gt_query: str, pred_query, tool_name: str) -> dict:
    if pred_query is None or str(pred_query).strip() in ('', 'nan', 'None'):
        return {"score": None, "reason": "query 없음 (검색 미실행 또는 오류)"}

    user_msg = f"""[슬라이드 내용]
{content}

[정답 쿼리] (tool: {tool_name})
{gt_query}

[평가 대상 쿼리]
{pred_query}"""

    try:
        response = judge_llm.invoke([
            SystemMessage(content=JUDGE_SYSTEM_PROMPT),
            HumanMessage(content=user_msg)
        ])
        text = response.content.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return {"score": None, "reason": f"JSON 파싱 실패: {response.content[:100]}"}
    except Exception as e:
        return {"score": None, "reason": f"오류: {str(e)}"}

print("Judge 준비 완료 (gpt-4o, 1~5점 척도)")

In [ ]:
# tool_called=True인 행만 평가 (query가 있어야 의미 있음)
df_search = df_result[df_result['tool_called'] == True].copy()
print(f"Query 평가 대상: {len(df_search)}행 (tool_called=True)\n")

for model_name in MODEL_NAMES:
    print(f"Judge 평가 중: {model_name}")
    col_q      = f"{model_name}_query"
    col_score  = f"{model_name}_query_score"
    col_reason = f"{model_name}_query_reason"

    scores, reasons = [], []

    for i, (idx, row) in enumerate(df_search.iterrows()):
        res = judge_query(
            content    = row['content'],
            gt_query   = row['query'],
            pred_query = row[col_q],
            tool_name  = row['tool_name']
        )
        scores.append(res.get('score'))
        reasons.append(res.get('reason'))
        print(f"  [{i+1:3d}/{len(df_search)}] score={res.get('score')}  {res.get('reason', '')[:60]}")

    df_result.loc[df_search.index, col_score]  = scores
    df_result.loc[df_search.index, col_reason] = reasons
    print(f"  ✅ 완료\n")

df_result.to_csv('./tool_execution_test_Dataset/model_evaluation_result.csv', index=False, encoding='utf-8-sig')
print("저장 완료 (query 점수 포함) → model_evaluation_result.csv")

In [ ]:
df_result = pd.read_csv('./tool_execution_test_Dataset/model_evaluation_result.csv')

# ── 모델별 평균 점수 ────────────────────────────────────────────────────────────
avg_scores = {}
score_dists = {}
for m in MODEL_NAMES:
    col = f"{m}_query_score"
    scores = pd.to_numeric(df_result[col], errors='coerce').dropna()
    avg_scores[m] = scores.mean() if len(scores) > 0 else 0
    score_dists[m] = scores.value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)

print("=== LLM별 Query 평균 점수 ===")
for m, s in avg_scores.items():
    print(f"  {m:30s}: {s:.3f}")

# ── 시각화 ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('LLM별 Query 품질 평가 (LLM-as-a-Judge, gpt-4o)', fontsize=14, fontweight='bold')

# 평균 점수 막대 그래프
ax = axes[0]
vals = [avg_scores[m] for m in MODEL_NAMES]
bars = ax.bar(model_labels, vals, color='mediumpurple', edgecolor='white')
ax.set_title('모델별 평균 점수 (1~5)', fontsize=12)
ax.set_ylim(0, 5.8)
ax.set_ylabel('평균 점수')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.axhline(y=5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{v:.2f}', ha='center', va='bottom', fontsize=9)

# 점수 분포 스택 막대
ax = axes[1]
score_colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
bottom = np.zeros(len(MODEL_NAMES))
for score_val, color in zip([1, 2, 3, 4, 5], score_colors):
    heights = [score_dists[m][score_val] for m in MODEL_NAMES]
    ax.bar(model_labels, heights, bottom=bottom, label=f'{score_val}점', color=color)
    bottom += np.array(heights)
ax.set_title('점수 분포 (스택)', fontsize=12)
ax.set_ylabel('행 수')
ax.legend(loc='upper right', fontsize=9)
ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig('./tool_execution_test_Dataset/query_score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()